# Explorando e Vetorizando dados textuais:

In [ ]:
import pandas as pd
df = pd.read_csv("C:\\Users\\User\\Desktop\\Data Science\\Machine Learning NLP\\dataset_avaliacoes.csv")
df.head()

In [ ]:
# verificar quantas linhas e avaliações temos no total
df.shape

In [ ]:
# Explorando as avaliações - quantidade de avaliações positivas e negativas:
df['sentimento'].value_counts('sentimento')

Verificando as Percepções das pessoas usuárias, selecionando uma avaliação positiva e uma negativa:

In [ ]:
# A experiência de quem adquiriu o celular foi tão satisfatória que as expectativas foram superadas, levando à recomendação do produto.
print('positiva \n')

df.avaliacao[0]

In [ ]:
# A insatisfação é evidente, uma vez que o produto não foi entregue e o valor não foi estornado.
print('negativa \n')

df.avaliacao[2]

# Transformando textos em dados numéricos

In [ ]:
# Transformando o texto em números
# Extração de todas as palavras presentes no texto e o cálculo da frequência de cada uma delas em cada avaliação. 
# Assim, obtemos um vetor que reflete a frequência das palavras de forma numérica.

from sklearn.feature_extraction.text import CountVectorizer

texto = ['Comprei um produto ótimo', 'Comprei um produto ruim']

vetorizar = CountVectorizer()
bag_of_words = vetorizar.fit_transform(texto)

In [ ]:
bag_of_words

In [ ]:
# Obtemos uma matriz com dimensões 2 por 5, ou seja, duas linhas e cinco colunas. 
# Essa matriz é do tipo esparsa, contendo muitos zeros, uma vez que várias palavras não estão presentes em todas as avaliações. 
# Isso porque há diversas palavras que não estarão presentes em várias avaliações.
# Para facilitar a visualização dessa matriz, podemos convertê-la em um dataframe utilizando a biblioteca Pandas. 
# Faremos isso para entendermos como a frequência foi calculada e o que mais foi realizado.
# Criamos uma variável chamada matriz_esparsa e utilizamos a biblioteca pandas com pd.DataFrame. 
# Em seguida, aplicamos o método sparse.from_spmatrix() para converter a matriz esparsa para o formato de dataframe.

#columns=vetorizar.get_feature_names_out() para capturar o nome das palavras

import pandas as pd

matriz_esparsa = pd.DataFrame.sparse.from_spmatrix(
    bag_of_words,
    columns=vetorizar.get_feature_names_out()
)

In [ ]:
matriz_esparsa

In [ ]:
# lowercase=False - garantir que as palavras não sejam convertidas para minúsculas
# max_features=50 - devido o número de colunas ter aumentato para 23.352, refletindo a quantidade de palavras. max_features=50 -  
# a matriz resultante incluirá as 50 palavras que aparecem com mais frequência na sacola de palavras, limitando assim a dimensão do conjunto de dados.

vetorizar = CountVectorizer(lowercase=False, max_features=50) 
bag_of_words = vetorizar.fit_transform(df.avaliacao)
print(bag_of_words.shape)

Agora, a matriz tem apenas 50 colunas.

In [ ]:
matriz_esparsa_avaliacoes = pd.DataFrame.sparse.from_spmatrix(
    bag_of_words,
    columns=vetorizar.get_feature_names_out()
)


In [ ]:
matriz_esparsa_avaliacoes 

 Temos uma matriz composta por 50 palavras, onde cada linha corresponde a uma avaliação. Quando uma palavra não está presente, o valor é 0; quando está presente, o valor é 1. Se a palavra aparecer mais de uma vez, o valor será maior, como na linha 4.

Assim, conseguimos transformar palavras em números.

# Separando o conjunto de dados:

Dividir os dados em dois conjuntos: 

um conjunto maior, destinado ao treinamento do algoritmo, que permitirá ao modelo aprender com esses dados e identificar padrões para realizar a classificação.

Também será utilizado uma pequena porção dos dados para teste, o que nos permitirá avaliar a performance do modelo em situações do mundo real, utilizando informações que ele ainda não processou. 
Essa etapa é crucial para verificar tanto a eficácia quanto a capacidade de generalização do modelo em relação a novos dados.

Separando os dados em treino e teste

In [ ]:
from sklearn.model_selection import train_test_split

x_treino, x_test, y_treino, y_test = train_test_split(bag_of_words, df.sentimento, random_state=4978)

# Algoritmo de classificação:

Escolheremos a regressão logística, um método amplamente utilizado em tarefas de classificação, como a análise de sentimento. Esse algoritmo calcula a probabilidade de uma instância pertencer a uma classe ou a outra.

Foi criado abaixo o modelo de classificação, alcançando uma acurácia de aproximadamente 0.7982, o que significa que acertamos quase 80% das previsões com os dados de teste. Embora nosso modelo esteja satisfatório, há espaço para aprimoramentos.

In [ ]:
from sklearn.linear_model import LogisticRegression

regressao_logistica = LogisticRegression()
regressao_logistica.fit(x_treino, y_treino)
acuracia = regressao_logistica.score(x_test, y_test)
print(acuracia)

# Explorando a Frequência e o Sentimento:

In [ ]:
from wordcloud import WordCloud

In [ ]:
todas_palavras = [texto for texto in df.avaliacao]

In [ ]:
todas_palavras

# Criação da Lista de Avaliações

In [ ]:
# lista contendo todas as avaliações que possuímos. Considerando que temos mais de 15 mil avaliações, precisamos aninhar todas em uma única lista. 
# Para isso, foi criado uma variável chamada todas_palavras 
#  em vez de mantê-las separadas por vírgulas, apra isso, foi concatenado abaixo as avaliações com um espaço em branco como separado
todas_palavras = ' '.join([texto for texto in df.avaliacao])

In [ ]:
todas_palavras

# Criação da Nuvem de Palavras

In [ ]:
nuvem_palavras = WordCloud().generate(todas_palavras)

In [ ]:
import matplotlib.pyplot as plt

nuvem_palavras = WordCloud(width=800, height=500, max_font_size=110, collocations=False).generate(todas_palavras)
plt.figure(figsize=(10,7))
plt.imshow(nuvem_palavras, interpolation='bilinear')
plt.axis('off')
plt.show()

Observamos que as palavras mais frequentes são "muito", "produto", "de", "o", "é", "não", "que", "recomendo", "rápido", "comprei" e "prazo". Mas esta visão abrange todas as avaliações, tanto positivas quanto negativas.

# Função de Filtragem por Sentimento

Anteriormente foi criado a nuvem de palavras, mas ela contém as avaliações de forma geral. 
Com isto, abaixo foi separado as avaliações positivas das negativas e gerado uma nuvem de palavras para cada uma. 

Isso permitirá entender o que leva as pessoas a elogiar ou criticar um produto, fornecendo insights valiosos para o e-commerce compreender essas avaliações.

In [ ]:
def nuvem_palavras(texto, coluna_texto, sentimento):
      # Filtrando as resenhas com base no sentimento especificado
  texto_sentimento = texto.query(f"sentimento == '{sentimento}'")[coluna_texto] # coluna_texto é o filtro que contém as avaliações positivas e negativas

  # Unindo todas as resenhas em uma única string 
  # As avaliações estão sendo separadas pelo sentimento, porém precisa concatenar em um texto corrido. Junção dos textos separados por espaço
  texto_unido = ' '.join(texto_sentimento)

  # Criando e exibindo a nuvem de palavras
  nuvem_palavras = WordCloud(width=800, height=500, max_font_size=110, collocations=False).generate(texto_unido)
  plt.figure(figsize=(10,7))
  plt.imshow(nuvem_palavras, interpolation='bilinear')
  plt.axis('off')
  plt.show()

# Nuvem de Avaliações Negativas

Vamos analisar as reclamações expressas nas avaliações negativas. Palavras como "não", "produto", "diferente", "devolução", "ruim", "defeito", "resposta" e "qualidade" aparecem com destaque. Isso indica insatisfação dos consumidores em relação à qualidade do produto, à comunicação com o e-commerce e ao processo de devolução. Com essa nuvem de palavras, o e-commerce pode identificar áreas que necessitam de melhorias.

In [ ]:
nuvem_palavras(df, 'avaliacao', 'negativo')

# Nuvem de Avaliações Positivas

Observamos que as pessoas elogiaram o produto por ser excelente, superando expectativas, destacando a qualidade, o custo, a rapidez na entrega, a beleza, a satisfação, o funcionamento e os benefícios.

In [ ]:
nuvem_palavras(df, 'avaliacao', 'positivo')

Conclusão:
Apesar das críticas à entrega nas avaliações negativas, também houve elogios à entrega e qualidade nas positivas. Com isso, a empresa pode identificar o que está funcionando bem e aprimorar ainda mais.

Foi criado duas nuvens de palavras: uma para sentimentos negativos e outra para positivos. 
Compreendemos tanto os pontos de insatisfação quanto os de apreciação dos consumidores. 

# Dividindo o texto em unidades menores

In [ ]:
todas_palavras

# Tokenização_

tokenização, que é usada para separar um texto em unidades menores, chamadas tokens. Esses tokens podem ser palavras, caracteres, entre outros, dependendo do nível de granularidade desejado. No nosso caso, trabalharemos com palavras. Utilizaremos essa técnica para separar o texto em palavras e obter a frequência delas.

In [ ]:
# Tokenização do Texto
import nltk
import ssl

try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

nltk.download('punkt')

In [ ]:
# Testando a Frequência com Frases de Exemplo
frases = ['um produto bom', 'um produto ruim']
frequencia = nltk.FreqDist(frases)
frequencia

Ao visualizar a frequência, obtemos um dicionário com as frases e suas frequências, mas queremos a frequência de cada palavra. Isso não funcionou porque ainda não fizemos a tokenização.

In [ ]:
# Implementação da Tokenização
from nltk import tokenize

frase = 'O produto é excelente e a entrega foi muito rápida!'

token_espaco = tokenize.WhitespaceTokenizer()
token_frase = token_espaco.tokenize(frase)
print(token_frase)

Ao imprimir o resultado, obtemos uma lista de tokens.

# Analisando a frequência das palavras


Tokenização das Avaliações

In [ ]:
# armazenando as palavras das avaliações
token_frase = token_espaco.tokenize(todas_palavras)
token_frase # lista com todas as palavras do nosso conjunto de dados.

# Cálculo da Frequência das Palavras

In [ ]:
frequencia = nltk.FreqDist(token_frase)
frequencia

In [ ]:
# Conversão para DataFrame
# converter esse dicionário em um dataframe
df_frequencia = pd.DataFrame({'Palavra': list(frequencia.keys()),
                              'Frequência': list(frequencia.values())})

In [ ]:
df_frequencia.head()


In [ ]:
# Identificação das Palavras Mais Frequentes
# ordenando o dataframe da maior para a menor frequência e selecionando as dez primeiras palavras
df_frequencia.nlargest(columns='Frequência', n=10)


In [ ]:
# Gráfico das Palavras Mais Frequentes
import seaborn as sns

plt.figure(figsize=(20,6))
ax = sns.barplot(data=df_frequencia.nlargest(columns='Frequência', n=20), x='Palavra', y='Frequência', color='gray')
ax.set(ylabel='Contagem')
plt.show()

O Gráfico mostra a frequência das 20 palavras mais comuns.

Gráfico de barras mostrando a contagem de frequência de palavras, com o eixo x representando as palavras e o eixo y representando a contagem. As barras em ordem decrescente de contagem representam as palavras "e", "o", "de", "a", "que", "não", "é", "produto", "com", "do", "muito", "para", "um", "no", "da", "o", "em", "uma", "na" e "mais".

Considerações Finais:

Observamos que palavras como "e", "o", "d", "a", "que", "não" são frequentes, mas não agregam muito valor semântico. 
Elas são artigos, conjunções e preposições, que não dizem muito sobre o sentimento das avaliações. 
Portanto, será interessante remover essas palavras irrelevantes para que nosso modelo possa aprender padrões mais significativos e classificar se uma avaliação é positiva ou negativa.

# Limpando e normalizando dados textuais

Necessidade de realizar um tratamento nos dados. 
A maioria das palavras mais frequentes não tem relevância ou valor semântico, pois não servem para diferenciar se um sentimento é positivo ou negativo.

Essas palavras são conhecidas como Stop Words (palavras de parada), comuns nos idiomas em geral, e estão muito frequentes, mas sem relevância para nós.

Manter essas Stop Words no modelo de classificação pode aumentar o ruído do modelo, diminuir sua eficácia e contribuir para aumentar sua dimensão. Se não limitarmos as palavras ao fazer o Bag of Words, aumentaremos a dimensão do modelo sem necessidade.

O ideal é focar no que é relevante. Por isso, realizaremos um tratamento para remover essas palavras.

In [ ]:
nltk.download('stopwords')

# Removendo stopwords
# Em seu interior, teremos uma lista contendo todas as palavras consideradas Stop Words em português, 
# como "a", "ao", "aquela", "aquele", "dele", "do", entre outras.

palavras_irrelevantes = nltk.corpus.stopwords.words('portuguese')
palavras_irrelevantes

In [ ]:
# criando uma Lista vazia chamada frase_processada, pois vamos processar as frases:
frase_processada = []

# Para cada opinião na coluna de avaliação, tokenizaremos o texto. Assim, informaremos cada opinião e dividiremos em tokens.
for opiniao in df['avaliacao']:
    palavras_texto = token_espaco.tokenize(opiniao)
    nova_frase = [palavra for palavra in palavras_texto if palavra not in palavras_irrelevantes]
    frase_processada.append(' '.join(nova_frase))
    

In [ ]:
# criando uma nova coluna no dataframe chamada tratamento_1. Nesse tratamento, removeremos as Stop Words.   
df['tratamento_1'] = frase_processada

In [ ]:
#  nova coluna chamada tratamento_1
df.head()

# Verificando mudanças na frase

Comparar alguma frase antes e depois da remoção de Stop Words para ver o que aconteceu

In [ ]:
# recolher a primeira avaliação no estado anterior:
df['avaliacao'][0]

In [ ]:
# recolher a mesma avaliação, após a remoção:
df['tratamento_1'][0] 

# Verificando impactos na acurácia

 verificaremos se esse tratamento impactou de alguma forma no modelo

In [ ]:
def classificar_texto(texto, coluna_texto, coluna_classificacao):
    vetorizar = CountVectorizer(lowercase=False, max_features=50)
    bag_of_words = vetorizar.fit_transform(texto[coluna_texto])
    X_treino, X_teste, y_treino, y_teste = train_test_split(bag_of_words, texto[coluna_classificacao], random_state=4978)
    regressao_logistica = LogisticRegression()
    regressao_logistica.fit(X_treino, y_treino)
    acuracia = regressao_logistica.score(X_teste, y_teste)
    return print(f"Acurácia do modelo com '{coluna_texto}': {acuracia * 100:.2f}%")

In [ ]:
classificar_texto(df, 'tratamento_1', 'sentimento')

Resultado: A acurácia do modelo com tratamento_1 foi de 81.09%, mostrando uma melhoria em relação à acurácia anterior, que estava próxima de 80%.

# Verificando impactos na frequência de palavras

Verificaremos como isso impacta no gráfico das palavras mais frequentes

In [ ]:
def grafico_frequencia(texto, coluna_texto, quantidade):
    todas_palavras = ' '.join([texto for texto in texto[coluna_texto]])
    token_espaco = tokenize.WhitespaceTokenizer()
    frequencia = nltk.FreqDist(token_espaco.tokenize(todas_palavras))
    df_frequencia = pd.DataFrame({"Palavra": list(frequencia.keys()),                                 "Frequência": list(frequencia.values())})
    df_frequencia = df_frequencia.nlargest(columns="Frequência", n=quantidade)
    plt.figure(figsize=(20,6))
    ax = sns.barplot(data=df_frequencia, x="Palavra", y ="Frequência", color='gray')
    ax.set(ylabel="Contagem")
    plt.show()

In [ ]:
grafico_frequencia(df, 'tratamento_1', 20)

Nota-se que o gráfico está diferente. A maior frequência passou a ser a palavra "produto", com "p" minúsculo.

Ainda precisa tratar algumas palavras, como "O" maiúsculo, "A" maiúsculo, e mais variações de "produto" com "P" maiúsculo e acompanhado de uma vírgula.

# Tratando a pontuação

In [ ]:
frase = 'Esse smartphone superou expectativas, recomendo'

# Executando o código, obtemos os seguintes tokens, sendo a vírgula um deles:
token_pontuacao = tokenize.WordPunctTokenizer()
token_frase = token_pontuacao.tokenize(frase)
print(token_frase)

In [ ]:
frase_processada = []

for opiniao in df['tratamento_1']:
    palavras_texto = token_pontuacao.tokenize(opiniao)
    nova_frase = [palavra for palavra in palavras_texto if 
    palavra.isalpha() and palavra not in palavras_irrelevantes]
    frase_processada.append(' '.join(nova_frase))

In [ ]:
df['tratamento_2'] = frase_processada

In [ ]:
# Nova coluna "tratamento_2"
# Comparando as colunas tratamento_1 e tratamento_2, nota-se a remoção de pontuação nas frases 
print(df.head())

# Verificando mudanças na frase

Comparando uma frase nos estados anterior e após o tratamento de remoção de pontuação. Utilizando a posição 10:

In [ ]:
# frase nos estados anterior:
df['tratamento_1'][10]

In [ ]:
# Verificando a mesma frase após o tratamento e notaremos que o texto fica sem pontuação:
df['tratamento_2'][10]

# Verificando impactos na frequência de palavras

verificando o impacto da alteração no gráfico de frequência das 20 palavras mais frequentes:

In [ ]:
grafico_frequencia(df, 'tratamento_2', 20)

O resultado mostra que não temos mais "produto" com vírgula no final, mas ainda temos "produto" com letras minúsculas e maiúsculas, além de acentuações.

Gráfico de frequência de palavras anterior. A palavra 'produto' ainda tem a maior contagem, chegando a quase 7000. As outras palavras sofreram variações e agora as mais frequentes após 'produto' são 'O', 'bom', 'entrega', 'qualidade', 'Não', 'recomendo', 'bem', 'A', 'prazo', 'Produto', 'chegou', 'recebi', 'veio', 'antes', 'pra', 'dia', 'ainda', 'compra' e 'excelente', variando de aproximadamente 2700 a menos de 1000.

# Removendo acentuação

In [ ]:
import unidecode

In [ ]:
# exemplo de uma frase com acento, onde tem um acento em "ótima" e uma cedilha em "preço". 
frase = 'Um aparelho ótima performance preço bem menor outros aparelhos marcas conhecidas performance semelhante'

teste = unidecode.unidecode(frase)
print(teste)
# Após a tratativa o resultado abaixo mostra a frase sem acentuação nas palavras "ótima" e em "preço". 

In [ ]:
# fazendo a mesma trativa porém agora com  o texto da variável tratamento_2
sem_acentos = [unidecode.unidecode(texto) for texto in df['tratamento_2']]
sem_acentos
# Após a tratativa o resultado abaixo mostra as frases sem acentuações


In [ ]:
# também deixaremos as Stop Words sem acento
stopwords_sem_acento = [unidecode.unidecode(texto) for texto in palavras_irrelevantes]
stopwords_sem_acento
# Após a tratativa o resultado abaixo mostra as stopwords sem acentuações

In [ ]:
#criando uma nova coluna para esse novo tratamento tratamento_3
df['tratamento_3'] = sem_acentos

frase_processada = []

for opiniao in df['tratamento_3']:
    palavras_texto = token_pontuacao.tokenize(opiniao)
    nova_frase = [palavra for palavra in palavras_texto if palavra not in stopwords_sem_acento]
    frase_processada.append(' '.join(nova_frase))

df['tratamento_3'] = frase_processada

In [ ]:
df.head()

Verificando mudanças na frase

In [ ]:
df['tratamento_2'][70]

In [ ]:
# Na tratamento_2 Havia acentos em "últimos", "útil", "confortável", "número", "durável", e cedilha em "digitação" e "espaço". 
# Na tratamento_3 as acentuações foram removidas:
df['tratamento_3'][70]

Verificando impactos na frequência de palavras

In [ ]:
# No resultado não apresenta mais nenhuma palavra com acentuação por exemplo a palavra Não que agora ficou Nao 
grafico_frequencia(df, 'tratamento_3', 20)

No entanto, ainda temos um problema importante: "produto" aparece duas vezes, com "P" maiúsculo e minúsculo, e ainda temos Stop Words como "O", "A" e "E" porque não uniformizamos o texto para ficar todo em letras minúsculas.

# Uniformizando o texto

In [ ]:
# exemplo uma frase com letras maiúsculas e minúsculas para ver como funciona esse tratamento
frase = "Bom produto otimo custo-beneficio Recomendo Confortavel bem acabado"

In [ ]:
print(frase.lower())

In [ ]:
# Aplicando a uniformização e criando uma nova coluna tratamento_4 com a mudança realizada com todas as palavras minúsculas 
frase_processada = []

for opiniao in df['tratamento_3']:
    opiniao = opiniao.lower()  #deixando todas a palabras em caixa baixa em letras minúsculas
    palavras_texto = token_pontuacao.tokenize(opiniao)
    nova_frase = [palavra for palavra in palavras_texto if palavra not in stopwords_sem_acento]
    frase_processada.append(' '.join(nova_frase))

df['tratamento_4'] = frase_processada

In [ ]:
df.head()

Verificando mudanças na frase

In [ ]:
df['tratamento_3'][3]

In [ ]:
df['tratamento_4'][3]

# Verificando a acurácia

In [ ]:
# Acurácia do modelo com 'tratamento_4': 83.75%
classificar_texto(df, 'tratamento_4', 'sentimento')

In [ ]:
# antes quando apenas foi removido as Stop Words, a acurácia do modelo foi para 81,9%:
classificar_texto(df, 'tratamento_1', 'sentimento')

Conclusão: 

Quando apenas foi removido as Stop Words, a acurácia foi de 81,09%. 

Agora, com o tratamento_4 e todos esses processos, alcançamos uma acurácia de 83,75%. 

Conseguimos melhorar ainda mais modelo.

# Simplificando as palavras

In [ ]:
nltk.download('rslp')
stemmer = nltk.RSLPStemmer()
stemmer.stem('gostei')

In [ ]:
stemmer.stem('gostado')

In [ ]:
stemmer.stem('gostou')

O resultado nos dois casos foi o mesmo, "gost", porque ambas palavras têm o mesmo radical. Assim, conseguimos reduzir palavras com o mesmo significado a um único radical.

In [ ]:
# Implementando o steammer no modelo
frase_processada = []
for opinion in df["tratamento_4"]:
    palavras_texto = token_pontuacao.tokenize(opinion)
    nova_frase = [stemmer.stem(palavra) for palavra in palavras_texto]
    frase_processada.append(" ".join(nova_frase))

df["tratamento_5"] = frase_processada

In [ ]:
# A nova coluna "tratamento_5" está diferente da anterior, e podemos comparar a avaliação antes e depois do stemming. 
# Para isso, usaremos um código para chamar um item das colunas 4 e 5.
df.head()

In [ ]:
print(df["tratamento_4"][3])

In [ ]:
print(df["tratamento_5"][3])

Antes, tínhamos "atendeu expectativas, achei luz ruim, nada dificulte funcionamento". Depois, "atende expect, ache luz, nada dificulte funcion". Algumas palavras permanecem inalteradas por serem curtas, mas as demais têm apenas o radical.

# Melhorando o modelo com o steammer

In [ ]:
# Acurácia do modelo com 'tratamento_5': 85.11%
classificar_texto(df, "tratamento_5", "sentimento")

Conclusão: Notamos que a acurácia aumentou de 83.75% para 85.11%, apenas com esse tratamento de deixar as palavras apenas com o radical. 

Com isso, melhoramos ainda mais nosso modelo.

# Aplicação do TF-IDF

TF-IDF técnica capaz de atribuir um peso às palavras, de modo que aquelas que diferenciam sentimentos tivessem pesos maiores, permitindo que o modelo utilizasse isso para classificar o sentimento como positivo ou negativorática.

calcula a frequência das palavras, utilizando uma normalização para obter um peso. Assim, atribui um peso maior às palavras mais relevantes, e nos ajuda a diferenciar os sentimentos.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer


frases = ['Comprei um ótimo produto', 'Comprei um produto péssimo']

tfidf = TfidfVectorizer(lowercase=False, max_features=50)
matriz = tfidf.fit_transform(frases)
pd.DataFrame(matriz.todense(),
             columns=tfidf.get_feature_names_out())

Na primeira linha, temos os valores dos pesos para a frase "Comprei um ótimo produto". Para "comprei", "produto" e "um", o peso é 0,44. Para "péssimo", o peso é zero, pois não está presente na frase. Já para "ótimo", o peso é 0,63, pois é a palavra que diferencia a frase.

Na segunda linha, para a frase "Comprei um produto péssimo", os pesos são semelhantes, mas "péssimo" tem um peso maior de 0,63. Portanto, essa técnica nos ajuda a identificar a relevância das palavras em uma frase.

# Aplicando a TF-IDF no modelo criado

In [ ]:
# Resultado com os dados ANTES de ser tratados com TF-IDF igual 73.48%
tfidf_bruto = tfidf.fit_transform(df["avaliacao"])
X_treino, X_teste, y_treino, y_teste = train_test_split(tfidf_bruto, df["sentimento"], random_state=4978)
regressao_logistica.fit(x_treino, y_treino)
acuracia_tfidf_bruto = regressao_logistica.score(X_teste, y_teste)
print(f"Acurácia do modelo: {acuracia_tfidf_bruto *100:.2f}%")

In [ ]:
# Resultado com os dados tratados com TF-IDF igual 85.14%
tfidf_tratados = tfidf.fit_transform(df['tratamento_5'])
X_treino, X_teste, y_treino, y_teste = train_test_split(tfidf_tratados, df['sentimento'], random_state=4978)
regressao_logistica.fit(X_treino, y_treino)
acuracia_tfidf_tratados = regressao_logistica.score(X_teste, y_teste)
print(f'Acurácia do modelo: {acuracia_tfidf_tratados *100:.2f}%')

Após executarmos, obtivemos uma acurácia de 85,14%. Houve um aumento, ainda que ligeiro, de 85,11% para 85,14%, mostrando que o TF-IDF é eficaz para nossos dados tratados.

# Capturando contextos

In [ ]:
# Ao executar, obtemos o seguinte resultado: três duplas de palavras - "comprei um", "um produto" e "produto ótimo".
from nltk import ngrams

frase = 'Comprei um produto ótimo'
frase_separada = token_espaco.tokenize(frase)
pares = ngrams(frase_separada, 2) # O número 2 representa os bi-grams, que trabalham com duas palavras
list(pares)

Agora conseguimos criar duplas de palavras que o modelo pode compreender melhor, ajudando na classificação. 
Agora ser~´a implementado isso nos dados e verificar se o Modelo melhora o resultado de 85,14%.

In [ ]:
# Melhorando a acurácia
tfidf_50 = TfidfVectorizer(lowercase=False, max_features=50, ngram_range=(1,2))
vetor_tfidf = tfidf_50.fit_transform(df['tratamento_5'])
X_treino, X_teste, y_treino, y_teste = train_test_split(vetor_tfidf, df['sentimento'], random_state=4978)
regressao_logistica.fit(X_treino, y_treino)
acuracia_tfidf_ngrams = regressao_logistica.score(X_teste, y_teste)
print(f'Acurácia do modelo com 50 features e ngrams: {acuracia_tfidf_ngrams * 100:.2f}%')

Conclusão do Resultado: Ao rodar o código, o modelo com 50 features e n-grams alcançou 85,22%, superando o modelo sem n-grams. No entanto, estamos usando apenas 50 features, o que pode ser insuficiente para muitos bigrams aparecerem logo de início.

# Explorando a quantidade de features na vetorização

In [ ]:
# Substituíndo tf_idf_50 por tf_idf_100 e foi alterado o max_features de 50 para 100. 
# No final, foi ajustado o print para mostrar a acurácia do modelo com 100 features e n-grams.
# Resultado de 88.21%

tfidf_100 = TfidfVectorizer(lowercase=False, max_features=100, ngram_range=(1,2))
vetor_tfidf = tfidf_100.fit_transform(df['tratamento_5'])
X_treino, X_teste, y_treino, y_teste = train_test_split(vetor_tfidf, df['sentimento'], random_state=4978)
regressao_logistica.fit(X_treino, y_treino)
acuracia_tfidf_ngrams = regressao_logistica.score(X_teste, y_teste)
print(f'Acurácia do modelo com 100 features e ngrams: {acuracia_tfidf_ngrams * 100:.2f}%')

In [ ]:
# Substituíndo tf_idf_100 por tf_idf_1000 e foi alterado o max_features de 100 para 114123. 
# No final, foi ajustado o print para mostrar a acurácia do modelo com 1000 features e n-grams.
# Resultado de 91.85%

tfidf_1000 = TfidfVectorizer(lowercase=False, max_features=114123, ngram_range=(1,2))
vetor_tfidf = tfidf_1000.fit_transform(df['tratamento_5'])
X_treino, X_teste, y_treino, y_teste = train_test_split(vetor_tfidf, df['sentimento'], random_state=4978)
regressao_logistica.fit(X_treino, y_treino)
acuracia_tfidf_ngrams = regressao_logistica.score(X_teste, y_teste)
print(f'Acurácia do modelo com 114123 features e ngrams: {acuracia_tfidf_ngrams * 100:.2f}%')

In [ ]:
# Executando sem limitadores com todas as features
# Resultado de 91.85%

tfidf = TfidfVectorizer(lowercase=False, ngram_range=(1,2))
vetor_tfidf = tfidf.fit_transform(df['tratamento_5'])
X_treino, X_teste, y_treino, y_teste = train_test_split(vetor_tfidf, df['sentimento'], random_state=4978)
regressao_logistica.fit(X_treino, y_treino)
acuracia_tfidf_ngrams = regressao_logistica.score(X_teste, y_teste)
print(f'Acurácia do modelo com todas as features e ngrams: {acuracia_tfidf_ngrams * 100:.2f}%')

In [ ]:
# Verificamos a dimensão do vetor
# 15.501 linhas e 114.123 colunas
vetor_tfidf.shape

In [ ]:

# Analisar a regressão logística: 
# A regressão logística permite verificar o peso atribuído às palavras, tanto para sentimentos positivos quanto negativos.
# Para verificar para o que o Modelo esta dando maior relevância / Peso

pesos = pd.DataFrame(
    regressao_logistica.coef_[0].T,
    index=tfidf_1000.get_feature_names_out()
)

In [ ]:
pesos.nlargest(50,0) # exibir do maior para o menor os 50 primeiros positivos da coluna 0

In [ ]:
pesos.nsmallest(50,0) # exibir do maior para o menor os 50 primeiros negativos da coluna 0

#  Testando o Modelo de Classificação

In [ ]:
# Salvando o modelo e o vetorizador
import joblib

joblib.dump(tfidf_1000, 'tfidf_vectorizer.pkl') # salvando o vetorizador
joblib.dump(regressao_logistica, 'modelo_regressao_logistica.pkl') # salvando o Modelo de Regressão Logística

In [ ]:
# Carregando o modelo e o vetorizador (Deserização)
tfidf = joblib.load('tfidf_vectorizer.pkl')
regressao_logistica = joblib.load('modelo_regressao_logistica.pkl')

# Criando uma função para processar novos dados

In [ ]:
palavras_irrelevantes = nltk.corpus.stopwords.words('portuguese')
token_pontuacao = tokenize.WordPunctTokenizer()
stemmer = nltk.RSLPStemmer()

def processar_avaliacao(avaliacao):
  # passo 1 - tokenizar as palavras
  tokens = token_pontuacao.tokenize(avaliacao)

  # passo 2 - remover as palavras irrelevantes
  frase_processada = [palavra for palavra in tokens if palavra.lower() not in palavras_irrelevantes]

  # passo 3 - remover tudo que não seja alfabético
  frase_processada = [palavra for palavra in frase_processada if palavra.isalpha()]

  # passo 4 - remover a acentuação com biblioteca Unidecode
  frase_processada = [unidecode.unidecode(palavra) for palavra in frase_processada]

  # passo 5 - Aplicado o stemming para obter o radical das palavras
  frase_processada = [stemmer.stem(palavra) for palavra in frase_processada]

  # retornar os dados processados
  return ' '.join(frase_processada) 

# Classificando novas avaliações

1 - Analisando a lista novas_avaliacoes

In [ ]:
# Novas avaliações para prever
novas_avaliacoes = ["Ótimo produto, super recomendo!",
                 "A entrega atrasou muito! Estou decepcionado com a compra",
                 "Muito satisfeito com a compra. Além de ter atendido as expectativas, o preço foi ótimo",
                 "Horrível!!! O produto chegou danificado e agora estou tentando fazer a devolução.",
                 '''Rastreando o pacote, achei que não fosse recebê-lo, pois, na data prevista, estava sendo entregue em outra cidade.
                 Mas, no fim, deu tudo certo e recebi o produto.Produto de ótima qualidade, atendendo bem as minhas necessidades e por
                 um preço super em conta.Recomendo.''']

In [ ]:
novas_avaliacoes_processadas = [processar_avaliacao(avaliacao) for avaliacao in novas_avaliacoes]

2 - Visualizando o resultado:

In [ ]:
novas_avaliacoes_processadas

# Classificando as novas avaliações

3 - Executar o modelo para fazer a predição(prever) das novas avaliações

In [ ]:
novas_avaliacoes_tfidf = tfidf.transform(novas_avaliacoes_processadas)

predicoes = regressao_logistica.predict(novas_avaliacoes_tfidf)

df_previsoes = pd.DataFrame({
    'Avaliação': novas_avaliacoes,
    'Sentimento previsto': predicoes
})

df_previsoes

Algumas avaliações são longas, então na tabela, temos trechos dos comentários seguidos de reticências, mas podemos convertê-la em uma tabela interativa para leitura completa.

Conclusão:
O Modelo classificou corretamente as novas avaliações, com uma acurácia de quase 92% nos dados de teste. 
Assim, temos um modelo pronto para ser usado pelo e-commerce, permitindo análise automática de sentimentos em tempo real. 

Dessa forma, conseguimos:

Otimizar soluções;

Melhorar o relacionamento com clientes;

Promover melhorias nos produtos;

E auxiliar em campanhas de marketing.